# **AI Job Market & Displacement Analyser**

# 01 — Data Collection & Inventory


| ID | File | Format |
|----|------|--------|
| A | dataset_a_jobs_in_data_2024.csv | CSV |
| B | dataset_b_AI_Job_Market_Trends_2026.csv | CSV |
| C | dataset_c_national_M2024_dl.xlsx | Excel |
| D | dataset_d_ai_job_trends_dataset.csv | CSV |
| E | dataset_e_Probability of Automation Dat....docx | Word |

---
## 0. Imports & paths

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
#for reading word document 

try:
    from docx import Document
    DOCX_OK = True
except ImportError:
    DOCX_OK = False
    print("python-docx library is not installed. Please install it to read .docx files.")

In [3]:
PROJECT_ROOT  = os.path.dirname(os.path.abspath(''))
DATA_DIR      = os.path.join(PROJECT_ROOT, 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── File paths matching actual files on disk ──────────────────────────────────
FILE_A = os.path.join(DATA_DIR, 'dataset_a_jobs_in_data_2024.csv')
FILE_B = os.path.join(DATA_DIR, 'dataset_b_AI_Job_Market_Trends_2026.csv')
FILE_C = os.path.join(DATA_DIR, 'dataset_c_national_M2024_dl.xlsx')
FILE_D = os.path.join(DATA_DIR, 'dataset_d_ai_job_trends_dataset.csv')

# Dataset E: auto-find the .docx (full name is truncated in Explorer)
docx_files = [f for f in os.listdir(DATA_DIR) if f.startswith('dataset_e') and f.endswith('.docx')]
FILE_E = os.path.join(DATA_DIR, docx_files[0]) if docx_files else None

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}\n')

# Verify every file exists before we proceed
for label, path in [('A (csv)', FILE_A), ('B (csv)', FILE_B),
                    ('C (xlsx)', FILE_C), ('D (csv)', FILE_D), ('E (docx)', FILE_E)]:
    status = 'FOUND' if (path and os.path.exists(path)) else 'NOT FOUND'
    print(f'  Dataset {label}: {status}')

Project root : d:\New folder\ai-job-market-analyser
Data dir     : d:\New folder\ai-job-market-analyser\data

  Dataset A (csv): FOUND
  Dataset B (csv): FOUND
  Dataset C (xlsx): FOUND
  Dataset D (csv): FOUND
  Dataset E (docx): FOUND


## 1. Datasets

In [4]:
df_A = pd.read_csv(FILE_A)

print(f'Shape  : {df_A.shape}')
print(f'Columns: {df_A.columns.tolist()}')
df_A.head(3)

Shape  : (14199, 12)
Columns: ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'work_setting', 'company_location', 'company_size', 'job_category']


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,work_setting,company_location,company_size,job_category
0,2024,Entry-level,Freelance,Applied Data Scientist,30000,USD,30000,United Kingdom,Remote,United Kingdom,M,Data Science and Research
1,2024,Executive,Full-time,Business Intelligence,230000,USD,230000,United States,In-person,United States,M,BI and Visualization
2,2024,Executive,Full-time,Business Intelligence,176900,USD,176900,United States,In-person,United States,M,BI and Visualization


In [5]:
print('Null counts:')
print(df_A.isnull().sum())
print()
print('dtypes:')
print(df_A.dtypes)

Null counts:
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
work_setting          0
company_location      0
company_size          0
job_category          0
dtype: int64

dtypes:
work_year             int64
experience_level        str
employment_type         str
job_title               str
salary                int64
salary_currency         str
salary_in_usd         int64
employee_residence      str
work_setting            str
company_location        str
company_size            str
job_category            str
dtype: object


## 2. Dataset B — AI Job Market Trends 2026 (CSV)

In [6]:
df_B = pd.read_csv(FILE_B)

print(f'Shape  : {df_B.shape}')
print(f'Columns: {df_B.columns.tolist()}')
df_B.head(3)

Shape  : (10345, 19)
Columns: ['job_id', 'job_title', 'company_size', 'company_industry', 'country', 'remote_type', 'experience_level', 'years_experience', 'education_level', 'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_month', 'job_posting_year', 'hiring_urgency', 'job_openings']


,job_id,job_title,company_size,company_industry,country,remote_type,experience_level,years_experience,education_level,skills_python,skills_sql,skills_ml,skills_deep_learning,skills_cloud,salary,job_posting_month,job_posting_year,hiring_urgency,job_openings
0,1,AI Engineer,Startup,Retail,Canada,Remote,Senior,2,Master,0,0,0,1,0,158322,6,2024,Low,4
1,2,Machine Learning Engineer,MNC,Technology,Australia,Hybrid,Mid,0,Bachelor,1,1,1,0,1,163666,11,2026,High,9
2,3,Machine Learning Engineer,MNC,Technology,Germany,Onsite,Mid,14,Master,1,0,1,0,1,158556,3,2026,High,9


In [7]:
# Verify the expected columns from your project brief
EXPECTED_B = [
    'job_title', 'company_size', 'company_industry', 'country',
    'remote_type', 'experience_level', 'salary',
    'skills_python', 'skills_sql', 'skills_ml'
]
missing = [c for c in EXPECTED_B if c not in df_B.columns]
if missing:
    print(f'WARNING — columns not found: {missing}')
    print(f'Actual columns: {df_B.columns.tolist()}')
else:
    print('All key columns present')

if 'salary' in df_B.columns:
    print(f'Salary range: {df_B["salary"].min():,.0f}  to  {df_B["salary"].max():,.0f}')

All key columns present
Salary range: 45,083  to  204,143


## 3. Dataset C — BLS OEWS National May 2024 (Excel)

In [8]:
xl_C = pd.ExcelFile(FILE_C)
print(f'Sheets in BLS file: {xl_C.sheet_names}')

# BLS files sometimes have a cover/notes sheet — find the real data sheet
df_C = None
for sheet in xl_C.sheet_names:
    temp = pd.read_excel(FILE_C, sheet_name=sheet)
    cols_str = ' '.join(str(c) for c in temp.columns).upper()
    if 'OCC' in cols_str or 'OCCUP' in cols_str:
        df_C = temp
        print(f'Using sheet: "{sheet}"')
        break

if df_C is None:
    # Fall back to first sheet
    df_C = pd.read_excel(FILE_C, sheet_name=xl_C.sheet_names[0])
    print('Using first sheet (auto-detect failed)')

print(f'Shape  : {df_C.shape}')
print(f'Columns: {df_C.columns.tolist()}')
df_C.head(3)

Sheets in BLS file: ['national_M2024_dl', 'Field Descriptions', 'UpdateTime', 'Filler']
Using sheet: "national_M2024_dl"
Shape  : (1403, 32)
Columns: ['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,23.8,37.89,60.44,29990,36730,49500,78810,125720,NaN,NaN
1,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,58.7,82.5,#,57010,79900,122090,171610,#,NaN,NaN
2,99,U.S.,1,US,0,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,50.48,81.01,#,47510,68800,104990,168490,#,NaN,NaN


In [9]:
for col in ['A_MEAN', 'A_MEDIAN', 'H_MEAN', 'H_MEDIAN']:
    if col in df_C.columns:
        n = (df_C[col].astype(str) == '#').sum()
        print(f'{col}: {n} suppressed (#) values')

A_MEAN: 0 suppressed (#) values
A_MEDIAN: 19 suppressed (#) values
H_MEAN: 0 suppressed (#) values
H_MEDIAN: 19 suppressed (#) values


## 4. Dataset D — AI Job Trends Dataset (CSV)

In [10]:
df_D = pd.read_csv(FILE_D)

print(f'Shape  : {df_D.shape}')
print(f'Columns: {df_D.columns.tolist()}')
df_D.head(3)

Shape  : (30000, 13)
Columns: ['Job Title', 'Industry', 'Job Status', 'AI Impact Level', 'Median Salary (USD)', 'Required Education', 'Experience Required (Years)', 'Job Openings (2024)', 'Projected Openings (2030)', 'Remote Work Ratio (%)', 'Automation Risk (%)', 'Location', 'Gender Diversity (%)']


,Job Title,Industry,Job Status,AI Impact Level,Median Salary (USD),Required Education,Experience Required (Years),Job Openings (2024),Projected Openings (2030),Remote Work Ratio (%),Automation Risk (%),Location,Gender Diversity (%)
0,Investment analyst,IT,Increasing,Moderate,42109.76,Master’s Degree,5,1515,6342,55.96,28.28,UK,44.63
1,"Journalist, newspaper",Manufacturing,Increasing,Moderate,132298.57,Master’s Degree,15,1243,6205,16.81,89.71,USA,66.39
2,Financial planner,Finance,Increasing,Low,143279.19,Bachelor’s Degree,4,3338,1154,91.82,72.97,Canada,41.13


In [11]:
# Identify risk and job title columns
risk_cols  = [c for c in df_D.columns if any(k in c.lower() for k in ['risk','auto','displace','impact','ai_'])]
title_cols = [c for c in df_D.columns if any(k in c.lower() for k in ['job','title','occup','role'])]
print(f'Risk columns  : {risk_cols}')
print(f'Title columns : {title_cols}')
print()
print('Null counts:')
print(df_D.isnull().sum())

Risk columns  : ['AI Impact Level', 'Automation Risk (%)']
Title columns : ['Job Title', 'Job Status', 'Job Openings (2024)']

Null counts:
Job Title                      0
Industry                       0
Job Status                     0
AI Impact Level                0
Median Salary (USD)            0
Required Education             0
Experience Required (Years)    0
Job Openings (2024)            0
Projected Openings (2030)      0
Remote Work Ratio (%)          0
Automation Risk (%)            0
Location                       0
Gender Diversity (%)           0
dtype: int64


## 5. Dataset E — Oxford Automation Probabilities (Word .docx)

In [12]:
if not DOCX_OK:
    raise ImportError('Install python-docx first — see warning in Cell 0')
if FILE_E is None:
    raise FileNotFoundError('No dataset_e .docx file found in data/ folder')

doc = Document(FILE_E)
print(f'File      : {os.path.basename(FILE_E)}')
print(f'Tables    : {len(doc.tables)}')
print(f'Paragraphs: {len(doc.paragraphs)}')

File      : dataset_e_Probability of Automation Data Set.docx
Tables    : 5
Paragraphs: 8


In [13]:
# Preview ALL tables — find which one has occupation + probability
for i, table in enumerate(doc.tables):
    rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
    print(f'\n--- Table {i}: {len(table.rows)} rows x {len(table.columns)} cols ---')
    for row in rows[:3]:
        print(row)


--- Table 0: 66 rows x 6 cols ---
['No.', 'Occupation', 'P(Auto)', 'Routineness', 'Complexity', 'Stratum']
['1', 'Chief executives, public administrators, and legislators', '0.015', '0.161', '0.979', '5']
['2', 'Financial managers', '0.069', '0.744', '0.961', '3']

--- Table 1: 67 rows x 6 cols ---
['No.', 'Occupation', 'P(Auto)', 'Routineness', 'Complexity', 'Stratum']
['66', 'Psychologists', '0.007', '0.638', '0.971', '3']
['67', 'Social scientists and sociologists, n.e.c.', '0.059', '0.066', '0.954', '3']

--- Table 2: 67 rows x 6 cols ---
['No.', 'Occupation', 'P(Auto)', 'Routineness', 'Complexity', 'Stratum']
['132', 'Mail clerks, outside of post office', '0.940', '0.957', '0.115', '1']
['133', 'Messengers', '0.940', '0.781', '0.130', '1']

--- Table 3: 67 rows x 6 cols ---
['No.', 'Occupation', 'P(Auto)', 'Routineness', 'Complexity', 'Stratum']
['198', 'Locksmiths and safe repairers', '0.770', '0.598', '0.418', '1']
['199', 'Repairers of mechanical controls and valves', '0.630',

In [14]:
# After previewing above, set TABLE_INDEX to the table with occupation + probability
TABLE_INDEX = 0   # <-- change this if needed

table = doc.tables[TABLE_INDEX]
rows  = [[cell.text.strip() for cell in row.cells] for row in table.rows]
df_E  = pd.DataFrame(rows[1:], columns=rows[0])   # row 0 = headers

print(f'Shape  : {df_E.shape}')
print(f'Columns: {df_E.columns.tolist()}')
df_E.head(5)

Shape  : (65, 6)
Columns: ['No.', 'Occupation', 'P(Auto)', 'Routineness', 'Complexity', 'Stratum']


,No.,Occupation,P(Auto),Routineness,Complexity,Stratum
0,1,"Chief executives, public administrators, and l...",0.015,0.161,0.979,5
1,2,Financial managers,0.069,0.744,0.961,3
2,3,Human resources and labour relations managers,0.006,0.355,0.913,3
3,4,"Managers and specialists in marketing, advert....",0.023,0.531,0.813,3
4,5,Managers in education and related fields,0.006,0.340,0.825,3


In [15]:
# Find and convert the probability column to numeric
prob_col = [c for c in df_E.columns if any(k in c.lower() for k in ['prob','risk','auto','suscept'])]
print(f'Probability column candidates: {prob_col}')

if prob_col:
    df_E[prob_col[0]] = pd.to_numeric(df_E[prob_col[0]], errors='coerce')
    print(df_E[prob_col[0]].describe())
else:
    print('No probability column found — check column names above')

Probability column candidates: ['P(Auto)']
count    65.000000
mean      0.177785
std       0.253795
min       0.004000
25%       0.012000
50%       0.066000
75%       0.240000
max       0.990000
Name: P(Auto), dtype: float64


## 6. Save all datasets as CSVs to data/processed/

In [16]:
save_map = {
    'dataset_A_raw.csv': df_A,
    'dataset_B_raw.csv': df_B,
    'dataset_C_raw.csv': df_C,
    'dataset_D_raw.csv': df_D,
    'dataset_E_raw.csv': df_E,
}

for fname, df in save_map.items():
    path = os.path.join(PROCESSED_DIR, fname)
    df.to_csv(path, index=False)
    print(f'Saved {fname}  ({df.shape[0]:,} rows x {df.shape[1]} cols)')

Saved dataset_A_raw.csv  (14,199 rows x 12 cols)
Saved dataset_B_raw.csv  (10,345 rows x 19 cols)
Saved dataset_C_raw.csv  (1,403 rows x 32 cols)
Saved dataset_D_raw.csv  (30,000 rows x 13 cols)
Saved dataset_E_raw.csv  (65 rows x 6 cols)


## 7. Inventory Summary

In [18]:
def summary(name, df, source, notes=''):
    return {
        'dataset'       : name,
        'source'        : source,
        'rows'          : df.shape[0],
        'columns'       : df.shape[1],
        'null_pct'      : round(df.isnull().sum().sum() / max(df.size,1) * 100, 2),
        'duplicate_rows': int(df.duplicated().sum()),
        'notes'         : notes
    }

inventory = pd.DataFrame([
    summary('A — Jobs in Data 2024',       df_A, 'Kaggle',   'salary, role, experience, country'),
    summary('B — AI Job Market 2026',      df_B, 'Kaggle',   'skills flags, salary, postings'),
    summary('C — BLS OEWS May 2024',       df_C, 'BLS.gov',  '830 occupations, mean/median wage'),
    summary('D — AI Job Trends 2024-2030', df_D, 'Kaggle',   'AI displacement, growth projections'),
    summary('E — Oxford Automation Probs', df_E, 'Mendeley', 'Frey-Osborne probability 0-1'),
])

print(f'Total records across all datasets: {inventory["rows"].sum():,}\n')
display(inventory)

inventory.to_csv(os.path.join(PROCESSED_DIR, '00_inventory.csv'), index=False)
print('\nInventory saved to data/processed/00_inventory.csv')


Total records across all datasets: 56,012



,dataset,source,rows,columns,null_pct,duplicate_rows,notes
0,A — Jobs in Data 2024,Kaggle,14199,12,0.00,5493,"salary, role, experience, country"
1,B — AI Job Market 2026,Kaggle,10345,19,0.00,0,"skills flags, salary, postings"
2,C — BLS OEWS May 2024,BLS.gov,1403,32,18.55,0,"830 occupations, mean/median wage"
3,D — AI Job Trends 2024-2030,Kaggle,30000,13,0.00,0,"AI displacement, growth projections"
4,E — Oxford Automation Probs,Mendeley,65,6,0.00,0,Frey-Osborne probability 0-1



Inventory saved to data/processed/00_inventory.csv
